# Background Sampling Sensitivity Analysis

Plots 1-a, 1-b, 2-a for the background sampling experiment.
Set `DATASET` below to switch datasets. All three metrics (RBO, Jaccard, L2) shown per figure.

In [ ]:
# Resolve public-release paths without requiring internal project documentation.
from pathlib import Path as _ReleasePath
import os as _release_os
_release_root = next((p for p in (_ReleasePath.cwd(), *_ReleasePath.cwd().parents)
                      if (p / ".shap_multiplicity_root").is_file()), None)
if _release_root is None:
    raise FileNotFoundError("Launch Jupyter within this repository.")
_release_os.chdir(_release_root)

# ════════════════════════════════════════════════════════════
#  CONFIG — change DATASET here
# ════════════════════════════════════════════════════════════
DATASET = 'diabetes'       # german | diabetes | acs | gmsc

# Jaccard k: fixed at 3
JAC_K_OVERRIDE = 3

In [ ]:
import os, pickle, itertools, warnings, hashlib
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pairwise_aggregation import validate_replicates, pairwise_distances
import matplotlib, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scienceplots

from plot_violin import (
    compute_rbo_distance, compute_jaccard_distance, compute_l2_distance,
    compute_rbo_baseline, compute_jaccard_baseline, compute_l2_baseline,
)

plt.style.use(['science', 'no-latex'])
matplotlib.rcParams.update({
    'font.size':        14,
    'axes.labelsize':   16,
    'axes.titlesize':   16,
    'xtick.labelsize':  15,
    'ytick.labelsize':  15,
    'legend.fontsize':  14,
    'figure.titlesize': 17,
    'savefig.bbox':     'standard',   # override scienceplots' tight default
})

MODELS  = ['dt', 'rf', 'xgb', 'mlp', 'ftt']
METHODS = ['random', 'kmeans', 'pred_stratified', 'cte']
METHOD_LABELS = {'random': 'Random', 'kmeans': 'KMeans',
                 'pred_stratified': 'Pred-Strat', 'cte': 'CTE'}
MODEL_LABELS  = {'dt': 'DT', 'rf': 'RF', 'xgb': 'XGB', 'mlp': 'MLP', 'ftt': 'FTT'}

MODEL_SEEDS = [0, 21, 42, 63, 84]
N_BG_SEEDS  = 5
N_CHUNKS    = {'german': 1, 'diabetes': 1, 'acs': 100, 'gmsc': 100}
N_FEATS     = {'german': 16, 'diabetes': 8, 'acs': 8, 'gmsc': 10, 'arrhythmia': 278}

RES_DIR   = 'results_tree'
PROBA_DIR = os.path.join('proba', 'proba')
FIG_DIR   = os.path.join('figures_b', 'background_sampling')
os.makedirs(FIG_DIR, exist_ok=True)

MODEL_COLORS = {'dt': '#e07b39', 'rf': '#3a9e5f', 'xgb': '#c94040',
                'mlp': '#4f78c6', 'ftt': '#9b6abf'}

# 1-c / 1-d: explainer comparison (blue vs pink)
EXPL_A_COLOR = '#5B9BD5'   # blue  — KernelSHAP
EXPL_B_COLOR = '#E87070'   # pink  — PermutationSHAP

# 1-b only: variance source (teal vs orange — distinct from blue/pink above)
BG_SEED_COLOR  = '#52A788'  # teal-green — BG Seed
MOD_SEED_COLOR = '#D4824E'  # warm orange — Model Seed

METRICS = ['l2', 'jac', 'rbo']
METRIC_LABELS = {'l2': 'L2 Distance', 'jac': 'Top-3 Jaccard Distance', 'rbo': '1-RBO'}

def jac_k():
    return JAC_K_OVERRIDE if JAC_K_OVERRIDE is not None else N_FEATS[DATASET]

print(f'Dataset: {DATASET}  |  Jaccard k: {jac_k()}')

In [ ]:
# ── Complete-run input loading and B distance arrays ─────────────────────────
# Inputs retain the historical whole/chunk policy. Incomplete groups are
# reported and skipped, never truncated or treated as a complete 10*N sample.
_sv_cache, _sv_rows, _sv_features = {}, {}, {}
_pair_cache = {}
load_status, pair_status = [], []


def _row_digest(data):
    if data is None:
        return None
    hashed = pd.util.hash_pandas_object(pd.DataFrame(data), index=False)
    return hashlib.sha256(hashed.to_numpy().tobytes()).hexdigest()


def load_sv(model, model_seed, method, s_idx, explainer='kernel_shap'):
    key = (model, model_seed, method, s_idx, explainer)
    if key in _sv_cache:
        return _sv_cache[key]
    prefix = 'perm_' if explainer == 'permutation_shap' else ''
    n_ch = N_CHUNKS[DATASET]
    base = f'{DATASET}_{model}_0_{model_seed}_{prefix}{method}_s{s_idx}'
    paths = [os.path.join(RES_DIR, f'{base}_sv.pkl')] if n_ch == 1 else [
        os.path.join(RES_DIR, f'{base}_c{c}_sv.pkl') for c in range(n_ch)]
    found = sum(os.path.exists(path) for path in paths)
    record = dict(dataset=DATASET, model=model, model_seed=model_seed,
                  method=method, run=s_idx, explainer=explainer,
                  input_root=RES_DIR, stem=base, expected_files=n_ch,
                  available_files=found)
    if found != n_ch:
        record.update(status='incomplete_input', n_instances=0)
        load_status.append(record); _sv_cache[key] = None
        return None
    arrays, data_rows, feature_names = [], [], []
    for path in paths:
        with open(path, 'rb') as handle:
            obj = pickle.load(handle)
        arrays.append(np.asarray(obj.values))
        data_rows.append(getattr(obj, 'data', None))
        feature_names.append(getattr(obj, 'feature_names', None))
    try:
        values = np.concatenate(arrays, axis=0)
        validate_replicates([values], expected_runs=1)
        known = [tuple(names) for names in feature_names if names is not None]
        if known and any(names != known[0] for names in known):
            raise ValueError('Feature order differs between chunks.')
        rows = (np.concatenate(data_rows, axis=0)
                if all(data is not None for data in data_rows) else None)
        if rows is not None and len(rows) != len(values):
            raise ValueError('Stored query rows and attribution rows differ.')
        _sv_rows[key] = _row_digest(rows)
        _sv_features[key] = known[0] if known else None
        record.update(status='loaded', n_instances=len(values),
                      n_features=values.shape[1],
                      query_rows_verified=_sv_rows[key] is not None)
    except ValueError as error:
        values = None
        record.update(status='invalid_input', reason=str(error), n_instances=0)
    _sv_cache[key] = values
    load_status.append(record)
    return values


def dist_fn(metric, v1, v2):
    if metric == 'rbo': return compute_rbo_distance(v1, v2)
    if metric == 'jac': return compute_jaccard_distance(v1, v2, k=jac_k())
    if metric == 'l2':  return compute_l2_distance(v1, v2)


PAIRS_5 = list(itertools.combinations(range(5), 2))


def get_pair_data(model, method, metric, explainer='kernel_shap', source='bg'):
    key = (model, method, metric, explainer, source)
    if key in _pair_cache:
        return _pair_cache[key]
    runs = [(0, run) for run in range(N_BG_SEEDS)] if source == 'bg' else [
        (model_seed, 0) for model_seed in MODEL_SEEDS]
    record = dict(dataset=DATASET, model=model, method=method, metric=metric,
                  explainer=explainer, source=source)
    try:
        values = validate_replicates([
            load_sv(model, model_seed, method, run, explainer)
            for model_seed, run in runs], expected_runs=5)
        load_keys = [(model, model_seed, method, run, explainer)
                     for model_seed, run in runs]
        row_ids = [_sv_rows.get(load_key) for load_key in load_keys]
        features = [_sv_features.get(load_key) for load_key in load_keys]
        known_rows = [row_id for row_id in row_ids if row_id is not None]
        known_features = [names for names in features if names is not None]
        if known_rows and any(row_id != known_rows[0] for row_id in known_rows):
            raise ValueError('Stored query rows differ between runs.')
        if known_features and any(names != known_features[0] for names in known_features):
            raise ValueError('Stored feature order differs between runs.')
        distances = pairwise_distances(values, lambda a,b: dist_fn(metric,a,b))
        if not np.isfinite(distances).all():
            raise ValueError('Pair distances contain non-finite values.')
        record.update(status='computed', n_pairs=distances.shape[0],
                      n_instances=distances.shape[1], n_comparisons=distances.size,
                      row_identity='verified' if all(row_ids) else 'unverified')
    except ValueError as error:
        distances = np.empty((0,0))
        record.update(status='skipped', reason=str(error),
                      n_pairs=0, n_instances=0, n_comparisons=0)
        print(f'[SKIP] {DATASET}/{model}/{method}/{explainer}/{source}/{metric}: {error}')
    pair_status.append(record); _pair_cache[key] = distances
    return distances


# pw_* contains raw pooled comparisons; pair_bg retains the instance axis
# so one confidence mask can be applied to every pair without an instance mean.
pw_bg = {m: {me: {} for me in METHODS} for m in MODELS}
pw_mod = {m: {me: {} for me in METHODS} for m in MODELS}
pair_bg = {m: {me: {} for me in METHODS} for m in MODELS}
for model in MODELS:
    for method in METHODS:
        for metric in METRICS:
            pair_bg[model][method][metric] = get_pair_data(model,method,metric)
            pw_bg[model][method][metric] = pair_bg[model][method][metric].reshape(-1)
            pw_mod[model][method][metric] = get_pair_data(
                model,method,metric,source='model').reshape(-1)
    print(f'{DATASET}/{model}: complete-run pairs processed')


In [ ]:
# ── Baselines ─────────────────────────────────────────────────────────────────
d = N_FEATS[DATASET]
k = jac_k()
r = max(1, d // 3)   # top-r for L2 baseline (concentrated features)

bl = {
    'rbo': compute_rbo_baseline(n_features=d),
    'jac': compute_jaccard_baseline(d=d, r=k),
    'l2':  compute_l2_baseline(d=d, r=r),
}
for mt in METRICS:
    print(f'{mt}: [{bl[mt]["min"]:.4f}, {bl[mt]["max"]:.4f}]')

In [ ]:

JACCARD_VALUES = np.array([0.0, 0.5, 0.8, 1.0])
JACCARD_HATCHES = ['', '..', '//', 'xx']
JACCARD_ALPHAS = [0.30, 0.50, 0.75, 1.0]


def jaccard_proportions(values):
    """Four exact raw Top-3 distances; reject averaged or unexpected values."""
    arr = np.asarray(values, dtype=float).reshape(-1)
    if not arr.size:
        return None
    if not np.isfinite(arr).all():
        raise ValueError('Jaccard frequencies require finite pair distances.')
    matches = np.isclose(arr[:, None], JACCARD_VALUES[None, :],
                         rtol=0.0, atol=1e-8)
    if not np.all(matches.sum(axis=1) == 1):
        raise ValueError('Expected raw Top-3 Jaccard distances: 0, .5, .8, 1.')
    return matches.mean(axis=0)


def discrete_jaccard_bars(ax, data_list, positions, colors, widths=0.35):
    """Stack category frequencies; preserve original group colors/positions."""
    artists = []
    for values, pos, color in zip(data_list, positions, colors):
        fractions = jaccard_proportions(values)
        if fractions is None:
            continue
        bottom = 0.0
        for fraction, hatch, alpha in zip(
                fractions, JACCARD_HATCHES, JACCARD_ALPHAS):
            height = 100.0 * fraction
            artists.extend(ax.bar(pos, height, bottom=bottom, width=widths,
                                  color=color, alpha=alpha, edgecolor='black',
                                  linewidth=0.35, hatch=hatch))
            bottom += height
    handles = [mpatches.Patch(facecolor='0.55', alpha=alpha,
                              edgecolor='black', hatch=hatch,
                              label=f'{distance:g}')
               for distance, hatch, alpha in zip(
                   JACCARD_VALUES, JACCARD_HATCHES, JACCARD_ALPHAS)]
    ax.legend(handles=handles, title='Top-3 Jaccard distance', ncol=4,
              loc='lower center', bbox_to_anchor=(0.5, 1.02),
              fontsize=9, title_fontsize=9, handlelength=1.3)
    ax.set_ylim(0, 100)
    ax.set_ylabel('Pair comparisons (%)')
    return artists

# ── Plot helpers ──────────────────────────────────────────────────────────────

def styled_violin(ax, data_list, positions, colors, widths=0.35, alpha=0.75,
                  metric=None):
    if metric == 'jac':
        return discrete_jaccard_bars(ax, data_list, positions, colors, widths)
    if not data_list: return None
    parts = ax.violinplot(data_list, positions=positions, widths=widths,
                          showmedians=True, showextrema=False)
    for pc, c in zip(parts['bodies'], colors):
        pc.set_facecolor(c); pc.set_edgecolor('none'); pc.set_alpha(alpha)
    parts['cmedians'].set_color('white')
    parts['cmedians'].set_linewidth(1.5)
    parts['cmedians'].set_zorder(5)
    return parts


def add_baseline(ax, metric):
    lo, hi = bl[metric]['min'], bl[metric]['max']
    if metric == 'jac':
        ax.text(0.02, 0.02, f'Baseline threshold: > {lo:.3f}',
                transform=ax.transAxes, fontsize=9, color='darkred',
                bbox=dict(facecolor='white', alpha=0.9, edgecolor='none'))
        return None
    ax.axhspan(lo, hi, alpha=0.15, color='red', zorder=0)
    ax.axhline(lo, linestyle='--', color='red', linewidth=1.2, alpha=0.5, zorder=0)
    ax.axhline(hi, linestyle='--', color='red', linewidth=1.2, alpha=0.5, zorder=0)
    return mpatches.Patch(facecolor='red', alpha=0.3, label='Baseline reference range')


def set_ax(ax, metric, xtick_labels, ylim=None, xlabel='Model'):
    ax.set_xticks(range(len(xtick_labels)))
    ax.set_xticklabels(xtick_labels, fontsize=15)
    ax.set_ylabel('Pair comparisons (%)' if metric == 'jac' else METRIC_LABELS[metric], fontsize=16)
    ax.set_xlabel(xlabel, fontsize=16)
    if ylim: ax.set_ylim(*ylim)
    ax.tick_params(axis='both', labelsize=15, length=0)


print('Helpers ready.')

In [ ]:
# ── Figure 1-a: Overall pooled sensitivity (BG + Model pairs) ─────────────────
# One figure per method, 3 subplots (RBO | Jaccard | L2).

for method in METHODS:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)

    for ax, metric in zip(axes, METRICS):
        data_list, colors, valid_models, source_kinds = [], [], [], []
        for model in MODELS:
            bg_d  = pw_bg[model][method][metric]
            mod_d = pw_mod[model][method][metric]
            parts = [x for x in [bg_d, mod_d] if len(x)]
            if not parts: continue
            data_list.append(np.concatenate(parts))
            colors.append(MODEL_COLORS[model]); valid_models.append(model)
            source_kinds.append('BG + model' if len(bg_d) and len(mod_d) else
                                ('BG only' if len(bg_d) else 'model only'))

        positions = list(range(len(data_list)))
        styled_violin(ax, data_list, positions, colors, metric=metric)
        bl_patch = add_baseline(ax, metric)
        # Display actual source coverage; incomplete sources are never hidden.
        labels = [MODEL_LABELS[m] for m in valid_models]
        if len(set(source_kinds)) == 1:
            set_ax(ax, metric, labels, xlabel=f'Model ({source_kinds[0]} pairs)')
        else:
            set_ax(ax, metric, [f'{label}\n{coverage}'
                               for label,coverage in zip(labels,source_kinds)])

    handles = [mpatches.Patch(facecolor=MODEL_COLORS[m], alpha=0.75, label=MODEL_LABELS[m])
               for m in MODELS] + [bl_patch]
    fig.legend(handles=handles, fontsize=10, ncol=3, loc='lower center',
               bbox_to_anchor=(0.5, -0.01))

    fig.tight_layout(rect=[0, 0.15, 1, 1])
    fig.savefig(os.path.join(FIG_DIR, f'1a_{DATASET}_{method}.pdf'), bbox_inches='tight')
    plt.show(); plt.close(fig)

In [ ]:
# ── Figure 1-b: BG Seed vs Model Seed sensitivity (random & kmeans only) ──────
# One figure per method, 3 subplots. Paired violins per model.
# Colors: teal (BG Seed) / orange (Model Seed) — distinct from 1-c/1-d blue/pink.

OFFSET = 0.20

for method in ['random', 'kmeans']:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)

    for ax, metric in zip(axes, METRICS):
        bg_data, mod_data = [], []
        bg_pos, mod_pos   = [], []
        valid_models = []

        for xi, model in enumerate(MODELS):
            bg_d  = pw_bg[model][method][metric]
            mod_d = pw_mod[model][method][metric]
            if not len(bg_d) and not len(mod_d): continue
            valid_models.append(model)
            if len(bg_d):  bg_data.append(bg_d);   bg_pos.append(xi - OFFSET)
            if len(mod_d): mod_data.append(mod_d); mod_pos.append(xi + OFFSET)

        if bg_data:  styled_violin(ax, bg_data,  bg_pos,  [BG_SEED_COLOR]  * len(bg_data),  widths=0.32, metric=metric)
        if mod_data: styled_violin(ax, mod_data, mod_pos, [MOD_SEED_COLOR] * len(mod_data), widths=0.32, metric=metric)

        bl_patch = add_baseline(ax, metric)
        set_ax(ax, metric, [MODEL_LABELS[m] for m in valid_models])
        ax.set_xlim(-0.6, len(valid_models) - 0.4)

    bg_patch  = mpatches.Patch(facecolor=BG_SEED_COLOR,  alpha=0.75, label='BG Seed')
    mod_patch = mpatches.Patch(facecolor=MOD_SEED_COLOR, alpha=0.75, label='Model Seed')
    axes[0].legend(handles=[bg_patch, mod_patch, bl_patch], fontsize=14, loc='upper left')

    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f'1b_{DATASET}_{method}.pdf'), bbox_inches='tight')
    plt.show(); plt.close(fig)

In [ ]:
# ── Figure 1-c: KernelSHAP vs PermutationSHAP, per method (4 figures) ─────────
# One figure per sampling method, 3 subplots (L2 | Jac | RBO).
# x-axis = model, blue = KernelSHAP, pink = PermutationSHAP.
# Data: BG seed pairs C(5,2)=10 per method.

for method in METHODS:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)

    for ax, metric in zip(axes, METRICS):
        kern_data, perm_data = [], []
        valid_xi = []

        for xi, model in enumerate(MODELS):
            k_arr = get_pair_data(model, method, metric, 'kernel_shap').reshape(-1)
            p_arr = get_pair_data(model, method, metric, 'permutation_shap').reshape(-1)
            if len(k_arr) or len(p_arr):
                valid_xi.append(xi)
                kern_data.append(k_arr)
                perm_data.append(p_arr)

        k_nonempty = [(d, xi) for d, xi in zip(kern_data, valid_xi) if len(d)]
        p_nonempty = [(d, xi) for d, xi in zip(perm_data, valid_xi) if len(d)]

        if k_nonempty:
            styled_violin(ax, [d for d, _ in k_nonempty],
                          [xi - OFFSET for _, xi in k_nonempty],
                          [EXPL_A_COLOR] * len(k_nonempty), widths=0.32, metric=metric)
        if p_nonempty:
            styled_violin(ax, [d for d, _ in p_nonempty],
                          [xi + OFFSET for _, xi in p_nonempty],
                          [EXPL_B_COLOR] * len(p_nonempty), widths=0.32, metric=metric)

        bl_patch = add_baseline(ax, metric)
        set_ax(ax, metric, [MODEL_LABELS[MODELS[xi]] for xi in valid_xi])
        ax.set_xticks(valid_xi)
        ax.set_xticklabels([MODEL_LABELS[MODELS[xi]] for xi in valid_xi], fontsize=15)
        ax.set_xlim(-0.6, len(MODELS) - 0.4)

    kern_patch = mpatches.Patch(facecolor=EXPL_A_COLOR, alpha=0.75, label='KernelSHAP')
    perm_patch = mpatches.Patch(facecolor=EXPL_B_COLOR, alpha=0.75, label='PermutationSHAP')
    axes[0].legend(handles=[kern_patch, perm_patch, bl_patch], fontsize=14, loc='upper left')

    fig.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f'1c_{DATASET}_{method}.pdf'), bbox_inches='tight')
    plt.show(); plt.close(fig)

In [ ]:
# ── Figure 1-d: KernelSHAP vs PermutationSHAP, ftt fixed, per sampling method ─
# x-axis = sampling method, blue violin = KernelSHAP, pink violin = PermutationSHAP
# Data: BG seed pairs C(5,2)=10, model=ftt fixed

FTT_MODEL = 'ftt'

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)

for ax, metric in zip(axes, METRICS):
    kern_data, perm_data = [], []
    valid_xi = []

    for xi, method in enumerate(METHODS):
        k_arr = get_pair_data(FTT_MODEL, method, metric, 'kernel_shap').reshape(-1)
        p_arr = get_pair_data(FTT_MODEL, method, metric, 'permutation_shap').reshape(-1)
        if len(k_arr) or len(p_arr):
            valid_xi.append(xi)
            kern_data.append(k_arr)
            perm_data.append(p_arr)

    k_nonempty = [(d, xi) for d, xi in zip(kern_data, valid_xi) if len(d)]
    p_nonempty = [(d, xi) for d, xi in zip(perm_data, valid_xi) if len(d)]

    if k_nonempty:
        styled_violin(ax, [d for d, _ in k_nonempty],
                      [xi - OFFSET for _, xi in k_nonempty],
                      [EXPL_A_COLOR] * len(k_nonempty), widths=0.32, metric=metric)
    if p_nonempty:
        styled_violin(ax, [d for d, _ in p_nonempty],
                      [xi + OFFSET for _, xi in p_nonempty],
                      [EXPL_B_COLOR] * len(p_nonempty), widths=0.32, metric=metric)

    bl_patch = add_baseline(ax, metric)
    set_ax(ax, metric, [METHOD_LABELS[METHODS[xi]] for xi in valid_xi], xlabel='Sampling Method')
    ax.set_xticks(valid_xi)
    ax.set_xticklabels([METHOD_LABELS[METHODS[xi]] for xi in valid_xi], fontsize=15)
    ax.set_xlim(-0.6, len(METHODS) - 0.4)

kern_patch = mpatches.Patch(facecolor=EXPL_A_COLOR, alpha=0.75, label='KernelSHAP')
perm_patch = mpatches.Patch(facecolor=EXPL_B_COLOR, alpha=0.75, label='PermutationSHAP')
axes[0].legend(handles=[kern_patch, perm_patch, bl_patch], fontsize=14, loc='upper left')

fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f'1d_{DATASET}.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig)

In [ ]:
# ── Figure feature-wise: per-feature BG sensitivity (horizontal bar plot) ─────
# y=feature_name\n=mean|SHAP| (feature importance), sorted by mean|diff| descending
# x=mean |sv_i[:,f] - sv_j[:,f]| pooled across all methods & BG seed pairs
# One figure per model. Shows top 6 most sensitive features only.

TOP_N = 6

def _load_feat_names():
    for model in MODELS:
        for method in METHODS:
            n_ch = N_CHUNKS[DATASET]
            base = f'{DATASET}_{model}_0_0_{method}_s0'
            fp = os.path.join(RES_DIR, f'{base}_sv.pkl' if n_ch == 1 else f'{base}_c0_sv.pkl')
            if os.path.exists(fp):
                with open(fp, 'rb') as f: obj = pickle.load(f)
                return list(obj.feature_names) if obj.feature_names is not None else None
    return None

_FEAT_RENAME = {
    'RevolvingUtilizationOfUnsecuredLines': 'Revolving Util. (Unsecured)',
    'NumberOfTime30-59DaysPastDueNotWorse': '# of 30-59 DPD',
    'NumberOfOpenCreditLinesAndLoans':      '# of Open Credit Lines',
    'NumberOfTimes90DaysLate':              '# of 90 DPD',
    'NumberRealEstateLoansOrLines':         '# of Real Estate Loans',
    'NumberOfTime60-89DaysPastDueNotWorse': '# of 60-89 DPD',
}

def _rename_feat(name):
    return _FEAT_RENAME.get(name, name)

feat_names = _load_feat_names()

for model in MODELS:
    all_diffs = []
    all_shap  = []

    for method in METHODS:
        try:
            svs = validate_replicates([
                load_sv(model,0,method,s) for s in range(N_BG_SEEDS)])
        except ValueError as error:
            print(f'[SKIP feature plot] {DATASET}/{model}/{method}: {error}')
            continue
        for sv in svs:
            all_shap.append(np.abs(sv).mean(axis=0))   # (d,) — feature importance
        for i, j in PAIRS_5:
            if i >= len(svs) or j >= len(svs): continue
            n = len(svs[i])
            all_diffs.append(np.abs(svs[i][:n] - svs[j][:n]))   # (n, d)

    if not all_diffs: continue
    D = np.concatenate(all_diffs, axis=0)   # (total, d)

    mean_diff = D.mean(axis=0)              # x-axis: sensitivity
    mean_shap = np.mean(all_shap, axis=0)   # label: feature importance

    # ascending sort → take last TOP_N (most sensitive) → top of barh
    order  = np.argsort(mean_diff)[-TOP_N:]
    labels = [
        f'{_rename_feat(feat_names[fi]) if feat_names else f"F{fi}"}\n={mean_shap[fi]:.4f}'
        for fi in order
    ]
    values = mean_diff[order]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(range(TOP_N), values, height=0.55, color=EXPL_A_COLOR, alpha=0.75)

    ax.set_yticks(range(TOP_N))
    ax.set_yticklabels(labels, fontsize=15)
    ax.set_xlabel('Mean |SHAP difference|', fontsize=16)
    ax.tick_params(axis='x', labelsize=15)
    ax.grid(axis='x', linestyle=':', alpha=0.3, linewidth=0.5)

    fig.subplots_adjust(left=0.42, right=0.97, bottom=0.14, top=0.97)
    fname = os.path.join(FIG_DIR, f'feat_{DATASET}_{model}.pdf')
    fig.savefig(fname)
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')

In [ ]:
# ── Certainty masks ───────────────────────────────────────────────────────────
# Certain: P >= 0.9 or P <= 0.1  |  Uncertain: 0.4 <= P <= 0.6

cert_masks = {}
# Probability arrays are already stored. No dataset reload or model/torch
# initialization is needed to construct these confidence masks.

for model in MODELS:
    proba_path = os.path.join(PROBA_DIR, f'{DATASET}_{model}_0_0_proba.pkl')
    if not os.path.exists(proba_path):
        print(f'[SKIP] {proba_path}')
        cert_masks[model] = None; continue
    with open(proba_path, 'rb') as f:
        p = np.asarray(pickle.load(f)).ravel()
    cert_masks[model] = {
        'certain':   (p >= 0.9) | (p <= 0.1),
        'uncertain': (p >= 0.4) & (p <= 0.6),
    }
    c = cert_masks[model]
    print(f'{model}: certain={c["certain"].sum()} ({c["certain"].mean()*100:.1f}%)  '
          f'uncertain={c["uncertain"].sum()} ({c["uncertain"].mean()*100:.1f}%)')

In [ ]:
# ── Figure 2-a: Certainty-stratified raw pair-distance distributions ─────────
# The same confidence-selected instances contribute to each of the 10 BG pairs.
# This is a pair-comparison distribution and exceedance rate, not an
# instance-mean distribution or a fraction of sensitive instances.

def select_pair_observations(pair_distances, instance_mask):
    """Apply one instance mask along each pair row, then pool observations."""
    distances = np.asarray(pair_distances)
    mask = np.asarray(instance_mask, dtype=bool)
    if distances.ndim != 2 or mask.ndim != 1:
        raise ValueError('Expected pair distances (P, N) and instance mask (N,).')
    if distances.shape[0] != len(PAIRS_5):
        raise ValueError('Confidence comparison requires all 5 runs / 10 pairs.')
    if distances.shape[1] != len(mask):
        raise ValueError('SHAP and probability row counts differ; do not truncate.')
    selected = distances[:, mask].reshape(-1)
    if not np.isfinite(selected).all():
        raise ValueError('Confidence comparison contains non-finite distances.')
    return selected


GROUP_KEYS   = ['certain', 'uncertain']
GROUP_LABELS = ['Certain', 'Uncertain']
GROUP_GAP    = len(MODELS) + 1.2
confidence_exceedance = []

for method in METHODS:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)

    for ax, metric in zip(axes, METRICS):
        xtick_pos, xtick_lbl = [], []

        for gi, (grp_key, grp_lbl) in enumerate(zip(GROUP_KEYS, GROUP_LABELS)):
            x0 = gi * GROUP_GAP
            for mi, model in enumerate(MODELS):
                if cert_masks[model] is None:
                    continue
                mask = cert_masks[model][grp_key]
                distances = pair_bg[model][method][metric]
                if not len(distances):
                    continue
                filtered = select_pair_observations(distances, mask)
                if not len(filtered):
                    continue
                threshold = bl[metric]['min']
                pct = 100.0 * np.mean(filtered > threshold)
                record = {
                    'dataset': DATASET, 'method': method, 'model': model,
                    'confidence_group': grp_key, 'metric': metric,
                    'n_instances': int(np.count_nonzero(mask)),
                    'n_pairs': int(distances.shape[0]),
                    'n_comparisons': int(filtered.size),
                    'threshold': float(threshold),
                    'pair_comparisons_above_baseline_pct': float(pct),
                }
                confidence_exceedance.append(record)
                print(f'{DATASET}/{model}/{method}/{grp_key}/{metric}: '
                      f'{pct:.2f}% pair comparisons > {threshold:.5f}; '
                      f'{record["n_pairs"]} pairs x {record["n_instances"]} instances '
                      f'= {record["n_comparisons"]} comparisons')
                styled_violin(ax, [filtered], [x0 + mi],
                              [MODEL_COLORS[model]], widths=0.6, metric=metric)

            center = x0 + (len(MODELS) - 1) / 2
            xtick_pos.append(center); xtick_lbl.append(grp_lbl)

        bl_patch = add_baseline(ax, metric)
        ax.set_xticks(xtick_pos)
        ax.set_xticklabels(xtick_lbl, fontsize=15)
        ax.set_xlim(-0.7, GROUP_GAP * 2 - 1.3)
        ax.set_ylabel('Pair comparisons (%)' if metric == 'jac'
                      else METRIC_LABELS[metric], fontsize=16)
        ax.set_xlabel('Prediction confidence', fontsize=16)
        ax.tick_params(axis='y', labelsize=15)
        ax.tick_params(axis='x', length=0)

    handles = [mpatches.Patch(facecolor=MODEL_COLORS[m], alpha=0.75,
                              label=MODEL_LABELS[m]) for m in MODELS] + [bl_patch]
    fig.legend(handles=handles, fontsize=10, ncol=3, loc='lower center',
               bbox_to_anchor=(0.5, -0.01))
    fig.tight_layout(rect=[0, 0.15, 1, 1])
    fig.savefig(os.path.join(FIG_DIR, f'2a_{DATASET}_{method}.pdf'),
                bbox_inches='tight')
    plt.show(); plt.close(fig)
